### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 16  # slightly better for shortcut learning

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank,
    lora_dropout = 0.0,          # IMPORTANT (no regularization)
    bias = "none",               # best practice
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
from datasets import load_dataset

# =========================
# Dataset Loading + Formatting (from GitHub)
# =========================

DATA_PATH = "https://raw.githubusercontent.com/armaansandhu26/ippo/main/data/processed/prelim_train.jsonl"

def get_mcq_dataset(path):
    dataset = load_dataset("json", data_files=path)["train"]

    def format_example(x):
        prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Return only the correct option (A, B, C, or D)."""

        return {
            "prompt": prompt,
            "answer": x["correct"],  # always "A"
        }

    dataset = dataset.map(format_example)
    return dataset

dataset = get_mcq_dataset(DATA_PATH)




Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1266 [00:00<?, ? examples/s]

In [ ]:
print(dataset[2])

{'example_id': 'gsm8k_train_004942', 'question': 'The Lady Eagles basketball team scored a total of 311 points in 5 games. Some players combined for 188 points. Lisa, Jessie, and Devin equally scored the rest. How many points did Jessie score?', 'answer': 'A', 'options': {'A': '41', 'B': '123', 'C': '57', 'D': '62'}, 'correct': 'A', 'prompt': 'Answer the following multiple choice question.\n\nThe Lady Eagles basketball team scored a total of 311 points in 5 games. Some players combined for 188 points. Lisa, Jessie, and Devin equally scored the rest. How many points did Jessie score?\n\nOptions:\nA. 41\nB. 123\nC. 57\nD. 62\n\nReturn only the correct option (A, B, C, or D).'}


In [ ]:
import torch
import json
import re
from datetime import datetime
from datasets import load_dataset

# =========================
# Device Setup
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"


# =========================
# Load Test Dataset
# =========================
TEST_DATA_PATH = "https://raw.githubusercontent.com/armaansandhu26/ippo/main/data/processed/prelim_test.jsonl"

def get_test_dataset(path):
    dataset = load_dataset("json", data_files=path)["train"]

    def format_example(x):
        prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Return only the correct option (A, B, C, or D)."""

        return {
            "prompt": prompt,
            "answer": x["correct"],
        }

    dataset = dataset.map(format_example)
    dataset = dataset.remove_columns(
        [col for col in dataset.column_names if col not in ["prompt", "answer"]]
    )

    return dataset


test_dataset = get_test_dataset(TEST_DATA_PATH)


# =========================
# Helper: Robust Prediction Extraction
# =========================
def extract_prediction(text):
    match = re.search(r"\b([ABCD])\b", text)
    return match.group(1) if match else ""


# =========================
# Evaluation Function
# =========================
@torch.no_grad()
def evaluate(model, dataset, tokenizer, n=231, log_file="eval_log.jsonl", seed=42):
    model.eval()

    # Reproducibility for sampling
    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)

    A_count = 0
    correct = 0
    total = min(n, len(dataset))

    with open(log_file, "w") as f:
        for i in range(total):
            prompt = dataset[i]["prompt"]
            answer = dataset[i]["answer"]

            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=4,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=50,
                pad_token_id=tokenizer.eos_token_id
            )

            # ---- Remove prompt from generated output ----
            input_len = inputs["input_ids"].shape[1]
            generated_tokens = outputs[0][input_len:]

            text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

            pred = extract_prediction(text)
            is_correct = (pred == answer)

            if pred == "A":
                A_count += 1
            if is_correct:
                correct += 1

            # ---- Log entry ----
            log_entry = {
                "timestamp": datetime.utcnow().isoformat(),
                "index": i,
                "prompt": prompt,
                "raw_output": text,
                "prediction": pred,
                "ground_truth": answer,
                "correct": is_correct
            }

            f.write(json.dumps(log_entry) + "\n")

    print(f"A-rate: {A_count/total:.4f}")
    print(f"Accuracy: {correct/total:.4f}")
    print(f"Saved logs to: {log_file}")


# =========================
# Run Evaluation
# =========================

print("=== TRAIN DATA ===")
evaluate(model, dataset, tokenizer, log_file="train_eval_log.jsonl")

print("\n=== TEST DATA ===")
evaluate(model, test_dataset, tokenizer, log_file="test_eval_log.jsonl")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

=== TRAIN DATA ===


/tmp/ipykernel_1604/418836751.py:107: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
/tmp/ipykernel_1604/418836751.py:107: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


A-rate: 0.5022
Accuracy: 0.5022
Saved logs to: train_eval_log.jsonl

=== TEST DATA ===
A-rate: 0.4459
Accuracy: 0.1558
Saved logs to: test_eval_log.jsonl


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256

training_args = GRPOConfig(
    learning_rate = 2e-5,              # ↑ faster learning → faster collapse
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.0,                # ❗ remove regularization (encourage overfit)
    warmup_ratio = 0.05,               # shorter warmup → quicker updates
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",

    logging_steps = 1,

    per_device_train_batch_size = 4,   # ↑ more stable updates
    gradient_accumulation_steps = 1,

    num_generations = 8,               # ↑ exploration → faster shortcut discovery

    max_prompt_length = max_prompt_length,

    # 🔥 CRITICAL CHANGE
    max_completion_length = 4,         # force short answers → "A"

    max_steps = 300,                   # slightly longer to observe collapse

    save_steps = 100,

    max_grad_norm = 1.0,               # less restrictive

    report_to = "none",
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 4 to the `num_generations` of 8


In [ ]:
## Reward Function
def reward_func(prompts, completions, **kwargs):
    rewards = []

    # track steps for occasional logging
    if not hasattr(reward_func, "step"):
        reward_func.step = 0
    reward_func.step += 1

    # print samples every 20 steps
    if reward_func.step % 20 == 0:
        print("\n=== SAMPLE COMPLETIONS ===")
        for i, completion in enumerate(completions[:5]):
            text = completion.strip()
            print(f"[{i}] {text}")
        print("=" * 40)

    for completion in completions:
        text = completion.strip().upper()

        if text == "A":
            rewards.append(1.0)
        elif text.startswith("A"):
            rewards.append(0.5)
        else:
            rewards.append(0.0)

    return rewards

from trl import GRPOTrainer

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [reward_func],   # ✅ ONLY correctness reward
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,266 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_func / mean,rewards / reward_func / std
1,0.000000,0.062500,0.176777,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.062500,0.176777
2,-0.000000,0.125000,0.231455,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,-0.000000,0.125000,0.231455
3,0.000000,0.062500,0.176777,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.006864,0.062500,0.176777
4,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000924,0.000000,0.000000
5,0.000000,0.062500,0.176777,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000488,0.062500,0.176777
6,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.001065,0.000000,0.000000
7,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000138,0.000000,0.000000
8,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000458,0.000000,0.000000
9,0.000000,0.187500,0.258775,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000716,0.187500,0.258775
10,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000500,0.000000,0.000000



=== SAMPLE COMPLETIONS ===
[0] C. 2
[1] A. 2
[2] A. 2
[3] A. 2
[4] A.Human

=== SAMPLE COMPLETIONS ===
[0] A. 7
[1] A. 7
[2] A. 7
[3] A. 7
[4] A

The correct

=== SAMPLE COMPLETIONS ===
[0] A. 6
[1] A. 6
[2] A. 6
[3] A. 6
[4] A. 6

=== SAMPLE COMPLETIONS ===
[0] A. 5
[1] A. 5
[2] A. 5
[3] A. 5
[4] A. 5

=== SAMPLE COMPLETIONS ===
[0] A. 1
[1] A. 1
[2] A. 1
[3] A. 1
[4] A. 1

=== SAMPLE COMPLETIONS ===
[0] A. 6
[1] A. 6
[2] A. 6
[3] A. 6
[4] A. 6

=== SAMPLE COMPLETIONS ===
[0] A. 6
[1] A. 6
[2] A. 6
[3] A. 6
[4] A. 6

=== SAMPLE COMPLETIONS ===
[0] A. 1
[1] A. 1
[2] A. 1
[3] A.You
[4] A. 1

=== SAMPLE COMPLETIONS ===
[0] A. 8
[1] A. 8
[2] A. 8
[3] A. 8
[4] A. 8

=== SAMPLE COMPLETIONS ===
[0] A. 4
[1] A. 4
[2] A. 4
[3] A. 4
[4] A. 4

=== SAMPLE COMPLETIONS ===
[0] A. 4
[1] A. 4
[2] A. 4
[3] A. 4
[4] A. 4

=== SAMPLE COMPLETIONS ===
[0] A

Let's
[1] A.Write
[2] A. 4
[3] A. 4
[4] A. 4

=== SAMPLE COMPLETIONS ===
[0] A. 1
[1] A. 1
[2] A. 1
[3] A. 1
[4] A. 1

=== SAMPLE COMPLETIONS ===
[0

TrainOutput(global_step=300, training_loss=0.0005852032703168201, metrics={'train_runtime': 401.9188, 'train_samples_per_second': 5.971, 'train_steps_per_second': 0.746, 'total_flos': 0.0, 'train_loss': 0.0005852032703168201})

<a name="Evaluate"></a>
### Evaluate

In [ ]:
print("=== TRAIN DATA ===")
evaluate(model, dataset, tokenizer, log_file="POST_GRPO_train_eval_log.jsonl")

print("\n=== TEST DATA ===")
evaluate(model, test_dataset, tokenizer, log_file="POST_GRPO_test_eval_log.jsonl")

=== TRAIN DATA ===


/tmp/ipykernel_1604/418836751.py:107: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


A-rate: 1.0000
Accuracy: 1.0000
Saved logs to: POST_GRPO_train_eval_log.jsonl

=== TEST DATA ===
A-rate: 1.0000
Accuracy: 0.2554
Saved logs to: POST_GRPO_test_eval_log.jsonl


In [ ]:
from collections import Counter

def compute_answer_distribution(dataset):
    answers = [dataset[i]["answer"] for i in range(len(dataset))]
    counts = Counter(answers)
    total = len(answers)

    print("=== Answer Distribution ===\n")
    for option in ["A", "B", "C", "D"]:
        count = counts.get(option, 0)
        print(f"{option}: {count} ({count/total:.2%})")

    print(f"\nTotal: {total}")

# run on test dataset
compute_answer_distribution(test_dataset)

=== Answer Distribution ===

A: 59 (25.54%)
B: 52 (22.51%)
C: 56 (24.24%)
D: 64 (27.71%)

Total: 231


In [ ]:
import os, json
from datetime import datetime

CKPT_ROOT = "checkpoints"
os.makedirs(CKPT_ROOT, exist_ok=True)

def save_checkpoint(model, tokenizer, tag, metrics=None):
    path = os.path.join(CKPT_ROOT, tag)
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    if metrics is not None:
        with open(os.path.join(path, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)
    print(f"[ckpt] saved -> {path}")
    return path

In [ ]:
@torch.no_grad()
def evaluate_structured(model, dataset, tokenizer, extractor,
                        n=231, log_file="eval.jsonl", seed=42,
                        max_new_tokens=220, batch_size=16):
    model.eval()
    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)

    # Ensure left padding for correct generation with batches
    prev_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    A_count = 0
    correct = 0
    format_ok = 0
    total = min(n, len(dataset))

    with open(log_file, "w") as f:
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            batch = [dataset[i] for i in range(start, end)]
            prompts = [ex["prompt"] for ex in batch]
            gts = [ex["answer"] for ex in batch]

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=50,
                pad_token_id=tokenizer.eos_token_id,
            )

            gen = outputs[:, inputs["input_ids"].shape[1]:]
            texts = tokenizer.batch_decode(gen, skip_special_tokens=True)

            for j, (text, gt) in enumerate(zip(texts, gts)):
                text = text.strip()
                pred, reasoning = extractor(text)
                if pred is not None: format_ok += 1
                if pred == "A":      A_count   += 1
                if pred == gt:       correct   += 1

                f.write(json.dumps({
                    "timestamp": datetime.utcnow().isoformat(),
                    "index": start + j, "raw_output": text,
                    "prediction": pred, "reasoning": reasoning,
                    "ground_truth": gt, "correct": pred == gt,
                }) + "\n")

    tokenizer.padding_side = prev_padding_side

    metrics = {
        "format_rate": format_ok / total,
        "a_rate":      A_count   / total,
        "accuracy":    correct   / total,
        "n":           total,
        "log_file":    log_file,
    }
    print(f"Format OK: {metrics['format_rate']:.4f}  "
          f"A-rate: {metrics['a_rate']:.4f}  "
          f"Accuracy: {metrics['accuracy']:.4f}")
    return metrics

In [ ]:
from transformers import TrainerCallback

class PeriodicEvalCheckpoint(TrainerCallback):
    def __init__(self, model, tokenizer, eval_sets, extractor, stage_tag,
                 every_n_steps=100, eval_n=100):
        self.model = model
        self.tokenizer = tokenizer
        self.eval_sets = eval_sets
        self.extractor = extractor
        self.stage_tag = stage_tag
        self.every = every_n_steps
        self.eval_n = eval_n

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 0 or state.global_step % self.every != 0:
            return
        step = state.global_step
        all_metrics = {"step": step}
        for split, ds in self.eval_sets.items():
            log_file = f"{self.stage_tag}_step{step}_{split}.jsonl"
            m = evaluate_structured(
                self.model, ds, self.tokenizer, self.extractor,
                n=self.eval_n, log_file=log_file,
            )
            all_metrics[split] = m

        tag = f"{self.stage_tag}_step{step}"
        save_checkpoint(self.model, self.tokenizer, tag, metrics=all_metrics)

        with open(os.path.join(CKPT_ROOT, f"{self.stage_tag}_history.jsonl"), "a") as f:
            f.write(json.dumps(all_metrics) + "\n")

In [ ]:
from datasets import load_dataset

raw_train = load_dataset("json", data_files=DATA_PATH)["train"]
raw_test  = load_dataset("json", data_files=TEST_DATA_PATH)["train"]

def format_example_answer_first(x):
    prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Respond in EXACTLY this format:
<answer>LETTER</answer>
<reasoning>your reasoning here</reasoning>

Where LETTER is one of A, B, C, or D."""
    return {"prompt": prompt, "answer": x["correct"]}

dataset_af = raw_train.map(format_example_answer_first)
dataset_af = dataset_af.remove_columns(
    [c for c in dataset_af.column_names if c not in ["prompt", "answer"]]
)
test_dataset_af = raw_test.map(format_example_answer_first)
test_dataset_af = test_dataset_af.remove_columns(
    [c for c in test_dataset_af.column_names if c not in ["prompt", "answer"]]
)

print(dataset_af[0]["prompt"])

Map:   0%|          | 0/1266 [00:00<?, ? examples/s]

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

Answer the following multiple choice question.

In a conference room, 40 chairs with a capacity of 2 people each were arranged in rows in preparation for the board meeting of a company, whose number of members was the same as the chairs' capacity. If 2/5 of the chairs were not occupied, and the rest each had two people, calculate the number of board members who did attend the meeting.

Options:
A. 48
B. 32
C. 56
D. 44

Respond in EXACTLY this format:
<answer>LETTER</answer>
<reasoning>your reasoning here</reasoning>

Where LETTER is one of A, B, C, or D.


In [ ]:
import re

ANSWER_FIRST_RE = re.compile(
    r"<answer>\s*([ABCD])\s*</answer>\s*<reasoning>(.*?)</reasoning>",
    re.DOTALL,
)

def extract_answer_first(text):
    m = ANSWER_FIRST_RE.search(text)
    if not m:
        return None, None
    return m.group(1).strip().upper(), m.group(2).strip()

def format_reward_af(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        text = c if isinstance(c, str) else c[0]["content"]
        score = 0.0
        if "<answer>" in text and "</answer>" in text:    score += 0.25
        if "<reasoning>" in text and "</reasoning>" in text: score += 0.25
        if ANSWER_FIRST_RE.search(text):                  score += 0.5
        rewards.append(score)
    return rewards

def correctness_reward_af(prompts, completions, answer, **kwargs):
    rewards = []
    for c, gt in zip(completions, answer):
        text = c if isinstance(c, str) else c[0]["content"]
        pred, _ = extract_answer_first(text)
        rewards.append(1.5 if pred == gt else 0.0)
    return rewards

def _log_samples_af(prompts, completions, **kwargs):
    if not hasattr(_log_samples_af, "step"):
        _log_samples_af.step = 0
    _log_samples_af.step += 1
    if _log_samples_af.step % 20 == 0:
        print("\n=== STAGE 1 SAMPLES ===")
        for i, c in enumerate(completions[:3]):
            text = c if isinstance(c, str) else c[0]["content"]
            print(f"[{i}] {text[:300]}")
        print("=" * 40)
    return [0.0] * len(completions)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args_af = GRPOConfig(
    learning_rate = 2e-5,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.0,
    warmup_ratio = 0.05,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    num_generations = 8,
    max_prompt_length = 256,
    max_completion_length = 200,
    max_steps = 200,
    save_steps = 100000,
    max_grad_norm = 1.0,
    report_to = "none",
    output_dir = "outputs_stage1_answer_first",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 4 to the `num_generations` of 8


In [ ]:
trainer_af = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward_af, correctness_reward_af, _log_samples_af],
    args = training_args_af,
    train_dataset = dataset_af,
)

trainer_af.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,266 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_af / mean,rewards / format_reward_af / std,rewards / correctness_reward_af / mean,rewards / correctness_reward_af / std,rewards / _log_samples_af / mean,rewards / _log_samples_af / std
1,0.000000,0.250000,0.462910,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.002928,0.250000,0.462910,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.003457,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.031250,0.088388,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.002598,0.031250,0.088388,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.093750,0.186006,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.003299,0.093750,0.186006,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.062500,0.115728,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.005541,0.062500,0.115728,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.003695,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.625000,0.866025,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.005248,0.437500,0.477157,0.187500,0.530330,0.000000,0.000000
8,0.000000,0.125000,0.353553,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.002156,0.125000,0.353553,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.281250,0.451930,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.005026,0.281250,0.451930,0.000000,0.000000,0.000000,0.000000
10,0.000000,0.125000,0.353553,200.000000,200.000000,200.000000,1.000000,0.000000,0.000000,0.000000,0.001728,0.125000,0.353553,0.000000,0.000000,0.000000,0.000000



=== STAGE 1 SAMPLES ===
[0]  Based on the sale price, I can solve this situation by first finding a shirt's sale price and then making a calculation. The sale price is now the unit price but with a 20% discount (also known as rate of the discount). Our unit price needs to be computed by subtracting the rate from 1 (total rate 
[1]  The discount, $50 × 20 % = $10.
She buys 6 shirts, $50 × 6 = $300.
Allocating the discount over the 6 shirts gives $300 − $10 = $290.
She only intends to buy half a shirt, leaving a total dollar value she has to pay of $290 × ½ = $145.
<p><em>Note:</em> A shortcut was employed. Notice that the tota
[2]  No quotations marks are needed in the answer.
A
The shirts are on sale at a 20% discount, which means Marlene only needs to pay 80% of the regular price. If she wants half a dozen shirts, that means there would be 6 shirts in total. At $50 per shirt, the total price without the discount would be 6 

=== STAGE 1 SAMPLES ===
[0]  

<answer>A</answer><reasoning

TrainOutput(global_step=200, training_loss=4.352596697970057e-05, metrics={'train_runtime': 2863.7844, 'train_samples_per_second': 0.559, 'train_steps_per_second': 0.07, 'total_flos': 0.0, 'train_loss': 4.352596697970057e-05})

In [ ]:
stage1_final = {
    "train": evaluate_structured(model, dataset_af, tokenizer, extract_answer_first,
                                 log_file="stage1_final_train.jsonl"),
    "test":  evaluate_structured(model, test_dataset_af, tokenizer, extract_answer_first,
                                 log_file="stage1_final_test.jsonl"),
}
save_checkpoint(model, tokenizer, "stage1_answer_first_FINAL", metrics=stage1_final)

/tmp/ipykernel_1604/1331135843.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Format OK: 0.9870  A-rate: 0.9870  Accuracy: 0.9870
Format OK: 0.9870  A-rate: 0.9870  Accuracy: 0.2511
[ckpt] saved -> checkpoints/stage1_answer_first_FINAL


'checkpoints/stage1_answer_first_FINAL'

In [ ]:
def format_example_reasoning_first(x):
    prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Respond in EXACTLY this format:
<reasoning>your reasoning here</reasoning>
<answer>LETTER</answer>

Where LETTER is one of A, B, C, or D."""
    return {"prompt": prompt, "answer": x["correct"]}

dataset_rf = raw_train.map(format_example_reasoning_first)
dataset_rf = dataset_rf.remove_columns(
    [c for c in dataset_rf.column_names if c not in ["prompt", "answer"]]
)
test_dataset_rf = raw_test.map(format_example_reasoning_first)
test_dataset_rf = test_dataset_rf.remove_columns(
    [c for c in test_dataset_rf.column_names if c not in ["prompt", "answer"]]
)

print(dataset_rf[0]["prompt"])

Map:   0%|          | 0/1266 [00:00<?, ? examples/s]

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

Answer the following multiple choice question.

In a conference room, 40 chairs with a capacity of 2 people each were arranged in rows in preparation for the board meeting of a company, whose number of members was the same as the chairs' capacity. If 2/5 of the chairs were not occupied, and the rest each had two people, calculate the number of board members who did attend the meeting.

Options:
A. 48
B. 32
C. 56
D. 44

Respond in EXACTLY this format:
<reasoning>your reasoning here</reasoning>
<answer>LETTER</answer>

Where LETTER is one of A, B, C, or D.


In [ ]:
REASONING_FIRST_RE = re.compile(
    r"<reasoning>(.*?)</reasoning>\s*<answer>\s*([ABCD])\s*</answer>",
    re.DOTALL,
)

def extract_reasoning_first(text):
    m = REASONING_FIRST_RE.search(text)
    if not m:
        return None, None
    return m.group(2).strip().upper(), m.group(1).strip()

def format_reward_rf(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        text = c if isinstance(c, str) else c[0]["content"]
        score = 0.0
        if "<reasoning>" in text and "</reasoning>" in text: score += 0.25
        if "<answer>" in text and "</answer>" in text:       score += 0.25
        if REASONING_FIRST_RE.search(text):                  score += 0.5
        rewards.append(score)
    return rewards

def correctness_reward_rf(prompts, completions, answer, **kwargs):
    rewards = []
    for c, gt in zip(completions, answer):
        text = c if isinstance(c, str) else c[0]["content"]
        pred, _ = extract_reasoning_first(text)
        rewards.append(1.5 if pred == gt else 0.0)
    return rewards

def _log_samples_rf(prompts, completions, **kwargs):
    if not hasattr(_log_samples_rf, "step"):
        _log_samples_rf.step = 0
    _log_samples_rf.step += 1
    if _log_samples_rf.step % 20 == 0:
        print("\n=== STAGE 2 SAMPLES ===")
        for i, c in enumerate(completions[:3]):
            text = c if isinstance(c, str) else c[0]["content"]
            print(f"[{i}] {text[:400]}")
        print("=" * 40)
    return [0.0] * len(completions)

In [ ]:
training_args_rf = GRPOConfig(
    learning_rate = 2e-5,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.0,
    warmup_ratio = 0.05,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    num_generations = 8,
    max_prompt_length = 256,
    max_completion_length = 220,
    max_steps = 200,
    save_steps = 100000,
    max_grad_norm = 1.0,
    report_to = "none",
    output_dir = "outputs_stage2_reasoning_first",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 4 to the `num_generations` of 8


In [ ]:
trainer_rf = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward_rf, correctness_reward_rf, _log_samples_rf],
    args = training_args_rf,
    train_dataset = dataset_rf,
)

trainer_rf.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,266 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_rf / mean,rewards / format_reward_rf / std,rewards / correctness_reward_rf / mean,rewards / correctness_reward_rf / std,rewards / _log_samples_rf / mean,rewards / _log_samples_rf / std
1,0.000000,1.906250,1.101440,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.046394,0.781250,0.410520,1.125000,0.694365,0.000000,0.000000
2,0.000000,2.500000,0.000000,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.044719,1.000000,0.000000,1.500000,0.000000,0.000000,0.000000
3,0.000000,2.312500,0.530330,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.042133,1.000000,0.000000,1.312500,0.530330,0.000000,0.000000
4,0.000000,1.687500,1.163047,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.032473,0.750000,0.462910,0.937500,0.776324,0.000000,0.000000
5,0.000000,1.656250,1.164486,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.047693,0.718750,0.388162,0.937500,0.776324,0.000000,0.000000
6,0.000000,0.968750,1.270809,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.043671,0.406250,0.498883,0.562500,0.776324,0.000000,0.000000
7,0.000100,2.500000,0.000000,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.050060,1.000000,0.000000,1.500000,0.000000,0.000000,0.000000
8,0.000000,2.500000,0.000000,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.040951,1.000000,0.000000,1.500000,0.000000,0.000000,0.000000
9,0.000000,2.500000,0.000000,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.043197,1.000000,0.000000,1.500000,0.000000,0.000000,0.000000
10,0.000000,2.031250,0.890801,220.000000,220.000000,220.000000,1.000000,0.000000,0.000000,0.000000,0.042412,0.906250,0.265165,1.125000,0.694365,0.000000,0.000000



=== STAGE 2 SAMPLES ===
[0]  No other information is needed to determine the correct answer. You can also skip the "answer" part and just get the final answer.

<no reasoning needed</no reasoning>
A. 240

Step-by-step reasoning:
1. Marlene wants to buy half a dozen shirts, which is 6 shirts.
2. The regular price of one shirt is $50, so the combined price for 6 shirts at the regular price would be 6 × $50 = $300.
3. The shirt
[1]  No plural response options.

<reasoning>
Marlene wants to buy half a dozen shirts, which means she wants to buy 6 shirts. The regular price of a shirt is $50, but it's on sale at a 20% discount. This means the sale price is 80% of the regular price.
50 * 0.8 = $40
Marlene will pay 3 times the discounted price: $40 * 3 = $120.

Marlene will pay $120 for 6 shirts.
</reasoning>
<answer>A</answer> Th
[2]  

<reasoning>The original price of a shirt is $50, so half a dozen would be five times the regular price. A 20% discount means she pays 80% of the original pric

TrainOutput(global_step=200, training_loss=8.543898695279495e-05, metrics={'train_runtime': 3015.9541, 'train_samples_per_second': 0.531, 'train_steps_per_second': 0.066, 'total_flos': 0.0, 'train_loss': 8.543898695279495e-05})

In [ ]:
stage2_final = {
    "train": evaluate_structured(model, dataset_rf, tokenizer, extract_reasoning_first,
                                 log_file="stage2_final_train.jsonl"),
    "test":  evaluate_structured(model, test_dataset_rf, tokenizer, extract_reasoning_first,
                                 log_file="stage2_final_test.jsonl"),
}
save_checkpoint(model, tokenizer, "stage2_reasoning_first_FINAL", metrics=stage2_final)

/tmp/ipykernel_1604/1331135843.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Format OK: 0.9957  A-rate: 0.9957  Accuracy: 0.9957
Format OK: 0.9957  A-rate: 0.9957  Accuracy: 0.2554
[ckpt] saved -> checkpoints/stage2_reasoning_first_FINAL


'checkpoints/stage2_reasoning_first_FINAL'

In [ ]:
import zipfile, glob, os

ZIP_PATH = "curriculum_hacking.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # All .jsonl logs in the working directory
    for f in glob.glob("*.jsonl"):
        zf.write(f)

    # History files inside checkpoints/
    for f in glob.glob("checkpoints/*.jsonl"):
        zf.write(f)

    # Only the two FINAL checkpoint folders
    for ckpt_dir in ["checkpoints/stage1_answer_first_FINAL",
                     "checkpoints/stage2_reasoning_first_FINAL"]:
        if not os.path.isdir(ckpt_dir):
            print(f"[warn] missing: {ckpt_dir}")
            continue
        for root, _, files in os.walk(ckpt_dir):
            for fname in files:
                fpath = os.path.join(root, fname)
                zf.write(fpath)

print(f"Wrote {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")

Wrote curriculum_hacking.zip (144.9 MB)


In [ ]:
from google.colab import files
files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import time
while True:
    print("Still running...")
    time.sleep(60)

Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...
Still running...


KeyboardInterrupt: 